In [ ]:
"""
pipeline/feature_pipeline.py
=============================
特徴量生成パイプライン（学習・推論共通）。

【このファイルが必要な理由】
    学習時と予測時で「まったく同じ計算式」を使わないと
    モデルが「違うデータが来た」と勘違いして精度が落ちる。
    ここに一本化することで「学習も予測も同じ処理を保証」する。

【ファイルの役割分担】
    features/calendar.py : カレンダー特徴量の計算ロジック
    features/lag.py      : ラグ特徴量の計算ロジック        ← build_features から呼ぶ
    features/rolling.py  : ローリング特徴量の計算ロジック  ← build_features から呼ぶ
    feature_pipeline.py  : 上記を正しい順番で呼び出す「指揮者」
                           + 推論時の1日更新ロジック（update_features_for_one_day）


【食品加工会社向け 特徴量セット】
    カレンダー系 : 月 / 曜日 / 土日フラグ / 祝日フラグ / 月末フラグ / 期末フラグ
    ラグ系       : lag_1 / lag_7 / lag_14（＋avail フラグ）
    ローリング系 : ma7 / ma28 / std7 / std7_valid
    入力データ   : price / stock（欠損補完済み）

【外部から呼び出す関数】
    build_features()              : 学習時に全データの特徴量をまとめて生成する
    update_features_for_one_day() : 予測時に1日分だけ特徴量を更新する
    get_feature_col_names()       : モデルへの入力列名リストを返す
"""

import numpy as np
import pandas as pd

from config import LAGS, ROLLING_WINDOWS
from features.calendar import add_calendar_features
from features.lag      import create_lags
from features.rolling  import create_rolling_features


# =========================================================
# 外部から呼び出す関数（公開 API）
# =========================================================

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    【学習時に使う】全データの特徴量をまとめて一括生成する。

    この関数でやること:
        ① 日付順に並び替える
        ② カレンダー特徴量（曜日・祝日・月末フラグなど）を追加  → features/calendar.py
        ③ ラグ特徴量（N日前の売上）を追加                     → features/lag.py
        ④ ローリング特徴量（移動平均・標準偏差）を追加          → features/rolling.py
           price / stock の欠損補完もここで行う

    各特徴量の計算ロジックは features/ 配下のファイルに書いてある。
    この関数は「正しい順番で呼び出す」ことだけを責任として持つ。

    Parameters
    ----------
    df : store / product / date / sales などを含む DataFrame

    Returns
    -------
    特徴量の列が追加された DataFrame
    """

    df = df.copy()

    # ① 店舗・商品・日付の順に並び替える
    # shift() や rolling() は「行の並び順」で計算するため、
    # 必ずこの順番に並べてからでないと正しい値が出ない
    df = (
        df.sort_values(["store", "product", "date"])
          .reset_index(drop=True)   # 並び替えた後は行番号を 0 から振り直す
    )

    # ② カレンダー特徴量（曜日・祝日・月末フラグなど）
    df = add_calendar_features(df)

    # ③ ラグ特徴量（N日前の売上）
    # 計算ロジックは features/lag.py の create_lags() に書いてある
    df = create_lags(df)

    # ④ ローリング特徴量（移動平均・標準偏差・price/stock 補完）
    # 計算ロジックは features/rolling.py の create_rolling_features() に書いてある
    df = create_rolling_features(df)

    return df




def get_feature_col_names(df: pd.DataFrame) -> list:
    """
    モデルへの入力として使う列名リストを返す。

    除外する列:
        store   : 店舗名そのもの（識別子であり特徴量ではない）
        product : 商品名そのもの（同上）
        date    : 日付そのもの（同上）
        sales   : 予測対象の正解ラベル（学習時のカンニング防止のため除外）

    残りの列がすべてモデルへの入力特徴量として使われる。
    例: ["月", "曜日", "土日か", "lag_1", "ma7", "std7", ...]
    """
    exclude_cols = {"store", "product", "date", "sales"}

    return [
        col
        for col in df.columns
        if col not in exclude_cols
    ]